In [ ]:
!date

### metagene plots around DMRs
- from /u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc/scripts/dmrs/A04*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
from scipy.stats import zscore
from sklearn.cluster import KMeans
from pybedtools import BedTool
import pysam
from scipy.ndimage import gaussian_filter

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'

donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
indir = f'{projdir}/csv/dmrs/rna_cluster_donor/meta/all_dmrs'
s = 'Start_C29'
dmr_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
dmr_flank.shape

In [ ]:
dmr_flank.head()

In [ ]:
s = 'Start'
samples = [f'{s}_{x}' for x in donors]
indir = f'{projdir}/csv/dmrs/rna_cluster_donor/meta/all_dmrs'

flanks = []
for i,s in enumerate(samples):
    d = donors[i]
    tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
    tmp_flank['plot_donor'] = d
    flanks.append(tmp_flank)
    
tmp_merged = pd.concat(flanks)
tmp_merged.shape

In [ ]:
hypo_merged = tmp_merged[tmp_merged['type'] == 'hypo']
hyper_merged = tmp_merged[tmp_merged['type'] == 'hyper']

In [ ]:
n_bins = 100
n_donors = len(donors)
hypo_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
hyper_merged['idx'] = np.tile(np.arange(n_bins), n_donors)

In [ ]:
d = 'C29'
tmp_hypo = hypo_merged[hypo_merged['plot_donor'] == d]
hypo_plot = tmp_hypo.melt(id_vars=['idx'], value_vars=samples)
hypo_plot['donor'] = [x.split('_')[1] for x in hypo_plot['variable']]
hypo_plot.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8, 4))

x_lab = 'idx'
y_lab = 'value'
hue = 'donor'

ax = sns.lineplot(data=hypo_plot, x=x_lab, y=y_lab, hue=hue, estimator='mean')
ax.set_xlabel('', fontsize=16)
ax.set_ylabel('', fontsize=16)
ax.set_title('', fontsize=20)

ax.tick_params(axis='x', labelsize=14, labelrotation=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

In [ ]:
d = 'C29'
tmp_hyper = hyper_merged[hyper_merged['plot_donor'] == d]
hyper_plot = tmp_hyper.melt(id_vars=['idx'], value_vars=samples)
hyper_plot['donor'] = [x.split('_')[1] for x in hyper_plot['variable']]
hyper_plot.head()

### plot other cell types' methylation in Start DMRs

In [ ]:
x_lab = 'idx'
y_lab = 'value'
hue = 'donor'

In [ ]:
n_bins = 100
n_donors = len(donors)
s = 'Start'
samples = [f'{s}_{x}' for x in donors]

flanks = []
for i,s in enumerate(samples):
    d = donors[i]
    tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
    tmp_flank['plot_donor'] = d
    flanks.append(tmp_flank)
tmp_merged = pd.concat(flanks)
hypo_merged = tmp_merged[tmp_merged['type'] == 'hypo']
hyper_merged = tmp_merged[tmp_merged['type'] == 'hyper']

hypo_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
hyper_merged['idx'] = np.tile(np.arange(n_bins), n_donors)

In [ ]:
n_bins = 100
n_donors = len(donors)
plot_samples = [f'Start_{x}' for x in donors]

for clust in tmp_order:
    tmp_samples = [f'{clust}_{x}' for x in donors]

    flanks = []
    for i,s in enumerate(tmp_samples):
        d = donors[i]
        tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
        tmp_flank['plot_donor'] = d
        flanks.append(tmp_flank)
        
    tmp_merged = pd.concat(flanks)
    hypo_merged = tmp_merged[tmp_merged['type'] == 'hypo'].copy()
    hyper_merged = tmp_merged[tmp_merged['type'] == 'hyper'].copy()

    hypo_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
    hyper_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
    
    fig, axes = plt.subplots(2, 2, figsize=(8, 4))
    for i,ax in enumerate(axes.flatten()):
        d = donors[i]
        tmp_hypo = hypo_merged[hypo_merged['plot_donor'] == d]
        hypo_plot = tmp_hypo.melt(id_vars=['idx'], value_vars=plot_samples)
        hypo_plot['donor'] = [x.split('_')[1] for x in hypo_plot['variable']]
        sns.lineplot(data=hypo_plot, x=x_lab, y=y_lab, hue=hue, estimator='mean', ax=ax)
        ax.set_xlabel('', fontsize=16)
        ax.set_ylabel('mCG', fontsize=12)
        ax.set_title(f'{clust}_{d} in Start donor-specific hypo DMRs' , fontsize=8)
        ax.set_ylim(0.3, 0.7)

        ax.legend_.remove()
        ax.tick_params(axis='x', labelsize=14, labelrotation=0)

        plt.subplots_adjust(wspace=0.2, hspace=0.6)

In [ ]:
n_bins = 100
n_donors = len(donors)
plot_samples = [f'Start_{x}' for x in donors]

for clust in tmp_order:
    tmp_samples = [f'{clust}_{x}' for x in donors]

    flanks = []
    for i,s in enumerate(tmp_samples):
        d = donors[i]
        tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
        tmp_flank['plot_donor'] = d
        flanks.append(tmp_flank)
        
    tmp_merged = pd.concat(flanks)
    hyper_merged = tmp_merged[tmp_merged['type'] == 'hyper'].copy()

    hyper_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
    
    fig, axes = plt.subplots(2, 2, figsize=(8, 4))
    for i,ax in enumerate(axes.flatten()):
        d = donors[i]
        tmp_hyper = hyper_merged[hyper_merged['plot_donor'] == d]
        hyper_plot = tmp_hyper.melt(id_vars=['idx'], value_vars=plot_samples)
        hyper_plot['donor'] = [x.split('_')[1] for x in hyper_plot['variable']]
        sns.lineplot(data=hyper_plot, x=x_lab, y=y_lab, hue=hue, estimator='mean', ax=ax)
        ax.set_xlabel('', fontsize=16)
        ax.set_ylabel('mCG', fontsize=12)
        ax.set_title(f'{clust}_{d} in Start donor-specific hyper DMRs' , fontsize=8)
        ax.set_ylim(0.3, 0.7)

        ax.legend_.remove()
        ax.tick_params(axis='x', labelsize=14, labelrotation=0)

        plt.subplots_adjust(wspace=0.2, hspace=0.6)

In [ ]:
# tmp_samples

In [ ]:
plot_order = ['Start', 'IPS']

In [ ]:
hypo_plot

In [ ]:
n_bins = 100
n_donors = len(donors)
plot_samples = [f'Start_{x}' for x in donors]

hypo_results = []
for clust in plot_order:
    tmp_samples = [f'{clust}_{x}' for x in donors]

    flanks = []
    for i,s in enumerate(tmp_samples):
        d = donors[i]
        tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
        tmp_flank['plot_donor'] = d
        flanks.append(tmp_flank)
        
    tmp_merged = pd.concat(flanks)
    hypo_merged = tmp_merged[tmp_merged['type'] == 'hypo'].copy()

    hypo_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
    
    fig, axes = plt.subplots(2, 2, figsize=(8, 4))
    for i,ax in enumerate(axes.flatten()):
        d = donors[i]
        tmp_hypo = hypo_merged[hypo_merged['plot_donor'] == d]
        hypo_plot = tmp_hypo.melt(id_vars=['idx'], value_vars=plot_samples)
        hypo_plot['cluster'] = clust
        hypo_plot['donor'] = [x.split('_')[1] for x in hypo_plot['variable']]
        sns.lineplot(data=hypo_plot, x=x_lab, y=y_lab, hue=hue, estimator='mean', ax=ax)
        ax.set_xlabel('', fontsize=16)
        ax.set_ylabel('mCG', fontsize=12)
        ax.set_title(f'{clust}_{d} in Start donor-specific hypo DMRs' , fontsize=8)
        ax.set_ylim(0.3, 0.7)

        ax.legend_.remove()
        ax.tick_params(axis='x', labelsize=14, labelrotation=0)

        plt.subplots_adjust(wspace=0.2, hspace=0.6)
        hypo_plot['plot_donor'] = d
        hypo_results.append(hypo_plot)

In [ ]:
n_bins = 100
n_donors = len(donors)
plot_samples = [f'Start_{x}' for x in donors]

hyper_results = []
for clust in plot_order:
    tmp_samples = [f'{clust}_{x}' for x in donors]

    flanks = []
    for i,s in enumerate(tmp_samples):
        d = donors[i]
        tmp_flank = pd.read_csv(f'{indir}/{s}.csv', sep='\t', index_col=0)
        tmp_flank['plot_donor'] = d
        flanks.append(tmp_flank)
        
    tmp_merged = pd.concat(flanks)
    hyper_merged = tmp_merged[tmp_merged['type'] == 'hyper'].copy()

    hyper_merged['idx'] = np.tile(np.arange(n_bins), n_donors)
    
    fig, axes = plt.subplots(2, 2, figsize=(8, 4))
    for i,ax in enumerate(axes.flatten()):
        d = donors[i]
        tmp_hyper = hyper_merged[hyper_merged['plot_donor'] == d]
        hyper_plot = tmp_hyper.melt(id_vars=['idx'], value_vars=plot_samples)
        hyper_plot['cluster'] = clust
        hyper_plot['donor'] = [x.split('_')[1] for x in hyper_plot['variable']]
        sns.lineplot(data=hyper_plot, x=x_lab, y=y_lab, hue=hue, estimator='mean', ax=ax)
        ax.set_xlabel('', fontsize=16)
        ax.set_ylabel('mCG', fontsize=12)
        ax.set_title(f'{clust}_{d} in Start donor-specific hyper DMRs' , fontsize=8)
        ax.set_ylim(0.3, 0.7)

        ax.legend_.remove()
        ax.tick_params(axis='x', labelsize=14, labelrotation=0)

        plt.subplots_adjust(wspace=0.2, hspace=0.6)
        hyper_plot['plot_donor'] = d
        hyper_results.append(hyper_plot)

In [ ]:
final_hypo = pd.concat(hypo_results)
final_hyper = pd.concat(hyper_results)

final_hypo['type'] = 'hypo'
final_hyper['type'] = 'hyper'
merged = pd.concat([final_hypo, final_hyper])
merged.shape

In [ ]:
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
merged.to_csv(f'{outdir}/donor_specific_dmrs_metagene.csv', sep='\t')

In [ ]:
!date